<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/04_multiphase_and_fields.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 4: Multi-Phase Transport and Field Visualisation

Many real microstructures contain more than two phases. In a lithium-ion battery electrode, for example, the active material, pore space, and carbon-binder domain (CBD) each have different transport properties. Treating the CBD as either fully blocking or fully open can introduce significant errors.

OpenImpala's effective diffusivity solver handles spatially varying diffusion coefficients $D(\mathbf{x})$, allowing each phase to be assigned its own conductivity. This tutorial demonstrates that capability and shows how to extract the solved field back into Python for visualisation.

**What you will learn:**
1. Create a 3-phase microstructure (solid matrix, pore, and binder).
2. Configure per-phase diffusivities via AMReX's `ParmParse`.
3. Solve the effective diffusivity cell problem with `openimpala.core`.
4. Extract the corrector field ($\chi$) into NumPy and plot flux pathways.

In [ ]:
# Install OpenImpala and visualization libraries
!pip install -q openimpala matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi
import openimpala.core as oic

# We also import pyAMReX to interact directly with the C++ data structures
import amrex.space3d as amrex

print(f"OpenImpala version {oi.__version__} loaded.")

## 1. Creating a 3-Phase Composite

Let's create a domain with three phases:
- **Phase 0 (Solid Matrix):** Completely insulating ($D = 0.0$)
- **Phase 1 (Electrolyte/Pore):** Highly conductive ($D = 1.0$)
- **Phase 2 (Binder/Carbon):** Partially conductive ($D = 0.1$)

In [ ]:
N = 64
micro = np.zeros((N, N, N), dtype=np.int32)  # Phase 0 everywhere

# Add a central "pore" channel (Phase 1)
micro[16:48, 16:48, :] = 1

# Add some "binder" blockages inside the pore (Phase 2)
micro[24:40, 24:40, 10:20] = 2
micro[20:44, 20:44, 40:50] = 2

plt.figure(figsize=(5, 5))
plt.imshow(micro[:, N//2, :], cmap='viridis')
plt.title("Slice of 3-Phase Composite (Y-Z plane)")
plt.xlabel("Z (Flow Direction)")
plt.ylabel("X")
# 0=Purple (Solid), 1=Teal (Pore), 2=Yellow (Binder)
plt.show()

## 2. Configuring Multi-Phase Physics & 3. Extracting the Solution Field

For multi-phase problems we need to pass phase-specific parameters to the underlying C++ engine. We do this using `amrex.ParmParse` inside the OpenImpala session.

After the solver finishes, we extract the corrector field ($\chi_z$) from the distributed AMReX `MultiFab` back into a NumPy array. Because AMReX objects require an active runtime, both the solve and the extraction must happen within the same `with oi.Session():` block.

> **Note:** This section uses lower-level APIs (`ParmParse`, `MultiFab`) from AMReX and OpenImpala's core module. These are intended for users who are comfortable with distributed HPC data structures. For most binary-phase problems, the high-level `oi.tortuosity()` API shown in earlier tutorials is sufficient.

In [ ]:
with oi.Session():
    # ==========================================
    # PART 1: Configure and Solve
    # ==========================================
    # 1. Define Multi-Phase settings in the AMReX global database
    # This tells the C++ backend: Phase 1 has D=1.0, Phase 2 has D=0.1
    pp = amrex.ParmParse("tortuosity")
    pp.addarr("active_phases", [1, 2])
    pp.addarr("phase_diffusivities", [1.0, 0.1])

    # 2. Convert NumPy array to AMReX iMultiFab
    # We use the facade helper to handle the domain decomposition
    geom, ba, dm, mf = oi.facade._numpy_to_imultifab(micro, max_grid_size=32)

    # 3. Initialize the Advanced Homogenization Solver
    # We tell it we are solving in the Z-direction (Direction.Z)
    print("Initializing Effective Diffusivity Solver...")
    solver = oic.EffectiveDiffusivityHypre(
        geom, ba, dm, mf,
        phase_id=1,                  # Primary reference phase
        dir=oic.Direction.Z,         # Flow direction
        solver_type=oic.EffDiffSolverType.FlexGMRES,
        results_path=".",
        verbose=1,
        write_plotfile=False
    )

    # 4. Run the solver
    print("Solving PDEs via HYPRE...")
    converged = solver.solve()
    print(f"Solver converged: {converged} in {solver.iterations} iterations.")

    # ==========================================
    # PART 2: Extracting the PDE Solution Field
    # ==========================================
    # Create an empty AMReX floating-point MultiFab to hold the result
    chi_field = amrex.MultiFab(ba, dm, 1, 1)

    # Ask the C++ solver to copy the HYPRE solution into our MultiFab
    solver.get_chi_solution(chi_field)

    # Convert the distributed AMReX MultiFab back into a single NumPy array
    # (Note: In a massive distributed MPI run, you wouldn't gather this to one node.
    # You would use `write_plotfile=True` and view it in ParaView/yt instead).
    chi_numpy = np.zeros((N, N, N), dtype=np.float64)

    for mfi in chi_field:
        # Get the valid box (no ghost cells) for this chunk
        bx = mfi.validbox()
        lo = bx.small_end
        hi = bx.big_end

        # Extract the local chunk as a 3D NumPy array
        local_arr = chi_field.array(mfi).to_numpy()

        # Place it into our global array
        chi_numpy[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1] = local_arr[:,:,:,0]

## 4. Visualising the Potential Field

With the solver output in NumPy, we can plot the corrector potential on a cross-section. Regions with steep gradients correspond to areas of high flux (and therefore high transport resistance). Notice how the gradient steepens where the flow is forced through the lower-conductivity binder phase.

In [ ]:
# Plot the potential field slice
plt.figure(figsize=(8, 6))

# Mask out the solid phase (Phase 0) so it shows up as white/empty
masked_chi = np.ma.masked_where(micro[:, N//2, :] == 0, chi_numpy[:, N//2, :])

img = plt.imshow(masked_chi, cmap='magma')
plt.colorbar(img, label="Corrector Potential ($\\chi_z$)")
plt.title("Internal Potential Field (Flow in Z-direction)")
plt.xlabel("Z (Flow Direction)")
plt.ylabel("X")
plt.axis('off')
plt.show()

## Next Steps

This tutorial showed how to set up and solve a multi-phase effective diffusivity problem and inspect the resulting field. The steep gradients around the binder regions indicate where transport resistance concentrates — information that is lost in a binary (solid/pore) approximation.

**Related tutorials:**
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Generate labelled datasets for machine learning.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Use OpenImpala as a cost-function evaluator in an optimisation loop.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Run on larger datasets with MPI.